# 01 · 数据获取与清洗

> 本 notebook 是《量化研究入门学习资料》第 X 章的可运行配套。
> 数据源：`data/csv/`。运行前请先执行 `python scripts/generate_data.py` 生成数据。

## 目标
从 `data/csv/prices.csv` 出发，完成与网页第 1 章一致的清洗管道（去重 → 前向填充 → 剔除退市后区间），并独立实现前/后复权算法。

In [ ]:
import pandas as pd, numpy as np

prices = pd.read_csv("data/csv/prices.csv", parse_dates=["date"])
basic = pd.read_csv("data/csv/stocks_basic.csv", parse_dates=["delist_date"])
dividends = pd.read_csv("data/csv/dividends.csv", parse_dates=["ex_date"])
print(f"原始 {len(prices):,} 行 × {prices['code'].nunique()} 只股票")

In [ ]:
def clean_pipeline(df, basic):
    """清洗：去重 → 排序 → close 前向填充 → 剔除退市日之后的记录"""
    df = df.drop_duplicates(["code", "date"], keep="last")
    df = df.sort_values(["code", "date"])
    df["close"] = df.groupby("code")["close"].ffill()          # 缺失值前向填充
    for _, r in basic.dropna(subset=["delist_date"]).iterrows():
        df = df.drop(df[(df["code"] == r["code"]) &
                        (df["date"] > r["delist_date"])].index)  # 退市后无数据
    return df

clean = clean_pipeline(prices, basic)
n_miss = int(clean["close"].isna().sum())
print(f"清洗后 {len(clean):,} 行；close 残留缺失 {n_miss} 行（应为 0）")
assert n_miss == 0, "清洗后不应残留缺失"

# 退市股校验：000180/000185/000190 最后记录日 == delist_date 前一日
for _, r in basic.dropna(subset=["delist_date"]).iterrows():
    last = clean[clean["code"] == r["code"]]["date"].max()
    print(f"  {r['code']} 最后数据日 {last.date()}（退市日 {r['delist_date'].date()}）")

In [ ]:
# 停牌与缺失统计（与网页 C5a 对照）
print("close 缺失率(原始):", round(float(prices["close"].isna().mean()), 4))
print("停牌比例:", round(float(clean["is_suspended"].mean()), 4))

In [ ]:
# ---- 复权计算（与生成脚本同款算法）----
df = clean.merge(dividends.rename(columns={"ex_date": "date"}),
                 on=["code", "date"], how="left")
df["cash"] = df["cash"].fillna(0.0)
df["bonus"] = df["bonus"].fillna(0.0)
df = df.sort_values(["code", "date"])
df["prev_close"] = df.groupby("code")["close"].shift(1)
# 除息日复权收益：(close + cash) × (1 + bonus) / prev_close − 1
df["r_adj"] = np.where(df["prev_close"].notna(),
                       (df["close"] + df["cash"]) * (1 + df["bonus"]) / df["prev_close"] - 1,
                       np.nan)
df["cum"] = df.groupby("code")["r_adj"].transform(lambda s: (1 + s.fillna(0)).cumprod())
df["adj_bwd"] = df["close"] * df["cum"]
last_close = df.groupby("code")["close"].transform("last")
last_bwd = df.groupby("code")["adj_bwd"].transform("last")
df["adj_fwd"] = df["adj_bwd"] * (last_close / last_bwd)

# 复权价合理性：除息日（cash>0 或 bonus>0）后复权收益应无跳空
sample = df[df["cash"] > 0].iloc[:1]
print("复权价计算完成 ✓ 前/后复权列已生成，见 df['adj_fwd'] / df['adj_bwd']")

In [ ]:
# ---- 一致性校验 ----
# 与 data/csv/prices.csv 的预计算复权列对照（round 2 位精度）
both = df.merge(clean[["code", "date", "adj_close_fwd", "adj_close_bwd"]],
                on=["code", "date"], suffixes=("", "_ref"))
ok_fwd = np.allclose(both["adj_fwd"].fillna(-1), both["adj_close_fwd"].fillna(-1), atol=0.011)
ok_bwd = np.allclose(both["adj_bwd"].fillna(-1), both["adj_close_bwd"].fillna(-1), atol=0.011)
print("前复权对照:", "PASS" if ok_fwd else "FAIL")
print("后复权对照:", "PASS" if ok_bwd else "FAIL")
assert ok_fwd and ok_bwd